# Data Analysis with SQL

Phân tích bằng SQL (DuckDB) để trả lời **RQ1** — nhu cầu bảo trì cây xanh biến động thế nào theo khu phố, theo ngày trong tuần, theo tháng và qua các năm — đồng thời tạo đặc trưng trễ và trung bình trượt cho Notebook 5, 7 và 8.

**Dữ liệu đầu vào** (từ Notebook 2):
- `SF_Tree_Weekly_Data_Cleaned.csv` → bảng `weekly` (mỗi dòng là một khu phố trong một tuần)
- `SF_311_Tree_Cases_Cleaned.csv` → bảng `cases` (mỗi dòng là một yêu cầu, dùng cho phân tích theo ngày trong tuần)

Các truy vấn 1–9 dùng biến mục tiêu chính `n` (đã loại yêu cầu trùng). Bảng `cases` có cả yêu cầu trùng nên được lọc bằng cờ `la_trung` để khớp với `n`.

In [1]:
# Nếu chưa có thư viện DuckDB, chạy 1 lần: pip install duckdb
import os
import duckdb
import pandas as pd

# Đọc 2 file đã làm sạch ở Notebook 2
df_weekly = pd.read_csv("../Data/processed/SF_Tree_Weekly_Data_Cleaned.csv", parse_dates=["week_start"])
df_cases = pd.read_csv("../Data/processed/SF_311_Tree_Cases_Cleaned.csv", parse_dates=["date", "week_start"])

print(df_weekly.shape)
print(df_cases.shape)

(25174, 15)
(132196, 12)


In [2]:
# Tạo database DuckDB và 2 bảng weekly, cases
con = duckdb.connect()
con.execute("CREATE TABLE weekly AS SELECT * FROM df_weekly")
con.execute("CREATE TABLE cases AS SELECT * FROM df_cases")

print("Đã tạo bảng SQL: weekly, cases")

Đã tạo bảng SQL: weekly, cases


In [3]:
# Kiểm tra bảng đã tạo
con.execute("""
    SELECT *
    FROM weekly
    LIMIT 5
""").df()

,nbhd,week_start,n,n_all,n_damaged,permits_active,PRCP,AWND,TMAX,so_cay,dbh_tb,tuoi_cay_tb,year,month,week_of_year
0,Bayview Hunters Point,2015-01-05,21,21,15,74.285714,0.0,2.3,15.485714,9030,6.0,15.816564,2015,1,2
1,Bayview Hunters Point,2015-01-12,7,7,5,78.428571,0.0,1.9,15.542857,9030,6.0,15.816564,2015,1,3
2,Bayview Hunters Point,2015-01-19,2,2,1,80.000000,0.0,2.0,16.828571,9030,6.0,15.816564,2015,1,4
3,Bayview Hunters Point,2015-01-26,5,6,0,85.142857,0.0,2.7,19.057143,9030,6.0,15.816564,2015,1,5
4,Bayview Hunters Point,2015-02-02,38,38,12,91.142857,51.0,11.0,18.400000,9030,6.0,15.816564,2015,2,6


### **1. Mục tiêu:** Khu phố nào có nhu cầu bảo trì cây xanh cao nhất?

In [4]:
print("\n5 khu phố có nhu cầu bảo trì cây cao nhất")

top_nbhd_sql = con.execute("""
    SELECT
        nbhd,
        CAST(SUM(n) AS INTEGER) AS tong_so_yeu_cau,
        ROUND(AVG(n), 2) AS tb_moi_tuan
    FROM weekly
    GROUP BY nbhd
    ORDER BY tong_so_yeu_cau DESC
    LIMIT 5;
""").df()

top_nbhd_sql


5 khu phố có nhu cầu bảo trì cây cao nhất


,nbhd,tong_so_yeu_cau,tb_moi_tuan
0,Mission,13013,21.19
1,West of Twin Peaks,7539,12.28
2,Bayview Hunters Point,6312,10.28
3,Sunset/Parkside,6060,9.87
4,Castro/Upper Market,5555,9.05


**Nhận xét:** Top 5 khu phố có nhu cầu cao nhất

**Vì sao dùng truy vấn này?** Truy vấn dùng `GROUP BY nbhd` để cộng tổng số yêu cầu và tính trung bình mỗi tuần của từng khu phố, sau đó sắp xếp giảm dần và lấy 5 khu phố cao nhất.

**Kết quả cần đọc:**
- **Mission** dẫn đầu rõ rệt với **13,013 yêu cầu** (khoảng **21 yêu cầu/tuần**).
- Gấp khoảng **1.7 lần** khu phố thứ 2 (West of Twin Peaks, 12.28/tuần) và **2.3 lần** khu phố thứ 5 (Castro/Upper Market, 9.05/tuần).
- Riêng 5 khu phố này chiếm khoảng **1/3 tổng nhu cầu** của cả thành phố.
- Đây là số yêu cầu tuyệt đối — khu phố nhiều cây thì nhiều yêu cầu hơn, cần đối chiếu với **truy vấn 3**.

### **2. Mục tiêu:** Khu phố nào có nhu cầu bảo trì cây xanh thấp nhất?

In [5]:
print("\n5 khu phố có nhu cầu bảo trì cây thấp nhất")

bottom_nbhd_sql = con.execute("""
    SELECT
        nbhd,
        CAST(SUM(n) AS INTEGER) AS tong_so_yeu_cau,
        ROUND(AVG(n), 2) AS tb_moi_tuan
    FROM weekly
    GROUP BY nbhd
    ORDER BY tong_so_yeu_cau ASC
    LIMIT 5;
""").df()

bottom_nbhd_sql


5 khu phố có nhu cầu bảo trì cây thấp nhất


,nbhd,tong_so_yeu_cau,tb_moi_tuan
0,Treasure Island,87,0.14
1,Lincoln Park,88,0.14
2,Presidio,96,0.16
3,McLaren Park,279,0.45
4,Seacliff,407,0.66


**Nhận xét:** Top 5 khu phố có nhu cầu thấp nhất

**Vì sao dùng truy vấn này?** Cùng cách làm với truy vấn 1, nhưng đổi `ORDER BY ... ASC` để lấy 5 khu phố có tổng số yêu cầu **thấp nhất**, dùng làm đối chứng với nhóm cao nhất.

**Kết quả cần đọc:**
- **Treasure Island** thấp nhất với chỉ **87 yêu cầu** trong 614 tuần — trung bình khoảng **7 tuần mới có 1 yêu cầu**.
- **Lincoln Park** (88) và **Presidio** (96) cũng gần như không có nhu cầu.
- So với Mission (13,013 yêu cầu), khu phố cao nhất gấp khoảng **150 lần** khu phố thấp nhất → nhu cầu phân bố **rất không đồng đều** giữa các khu phố.
- Phần lớn nhóm này là **khu công viên hoặc khu ít dân cư** (Lincoln Park, Presidio, McLaren Park, đảo Treasure Island), nơi có ít cây đường phố — cần đối chiếu với số cây ở **truy vấn 3**.

### **3. Mục tiêu:** Khi tính theo số cây, khu phố nào có nhu cầu bảo trì cao nhất?

In [6]:
print("\n10 khu phố có số yêu cầu trên 1.000 cây cao nhất")

per_tree_sql = con.execute("""
    SELECT
        nbhd,
        MAX(so_cay) AS so_cay,
        CAST(SUM(n) AS INTEGER) AS tong_so_yeu_cau,
        ROUND(SUM(n) * 1000.0 / MAX(so_cay), 1) AS yeu_cau_tren_1000_cay
    FROM weekly
    GROUP BY nbhd
    HAVING MAX(so_cay) >= 1000      -- bỏ khu phố quá ít cây (chủ yếu là công viên)
    ORDER BY yeu_cau_tren_1000_cay DESC
    LIMIT 10;
""").df()

per_tree_sql


10 khu phố có số yêu cầu trên 1.000 cây cao nhất


,nbhd,so_cay,tong_so_yeu_cau,yeu_cau_tren_1000_cay
0,Tenderloin,1446,2442,1688.8
1,Mission,9115,13013,1427.6
2,Hayes Valley,3399,4688,1379.2
3,South of Market,3142,4262,1356.5
4,Nob Hill,1895,2519,1329.3
5,Haight Ashbury,3175,3798,1196.2
6,Castro/Upper Market,4812,5555,1154.4
7,Inner Sunset,3558,3698,1039.3
8,Outer Mission,2757,2854,1035.2
9,Russian Hill,2485,2425,975.9


**Nhận xét:** Nhu cầu tính trên 1.000 cây

**Vì sao dùng truy vấn này?** Khu phố nhiều cây thì đương nhiên nhiều yêu cầu, nên chia tổng số yêu cầu cho số cây (`so_cay`) để so sánh công bằng hơn. Chỉ xét khu phố có từ **1.000 cây** trở lên (`HAVING`), vì các khu chủ yếu là công viên (Treasure Island, Lincoln Park chỉ có vài cây đường phố) cho tỷ lệ cao bất thường do mẫu số quá nhỏ. Số cây được tính tại mốc 03/01/2022 (Notebook 2).

**Kết quả cần đọc:**
- **Tenderloin** đứng đầu (**1,689 yêu cầu/1.000 cây**), vượt cả Mission (1,428).
- Top 10 chủ yếu là **các khu trung tâm đông đúc**: Hayes Valley, South of Market, Nob Hill, Haight Ashbury...
- **West of Twin Peaks, Bayview Hunters Point, Sunset/Parkside** (thứ 2–4 ở truy vấn 1) **không còn trong top 10** → các khu này nhiều yêu cầu chủ yếu vì **nhiều cây**, không phải vì cây dễ hỏng hơn.

### **4. Mục tiêu:** Khu phố cao nhất có nhu cầu gấp bao nhiêu lần khu phố thấp nhất?

In [7]:
print("\nMức chênh lệch giữa khu phố cao nhất và thấp nhất")

gap_sql = con.execute("""
    WITH nbhd_stats AS (
        SELECT
            nbhd,
            MAX(so_cay) AS so_cay,
            CAST(SUM(n) AS INTEGER) AS tong_so_yeu_cau,
            SUM(n) * 1000.0 / MAX(so_cay) AS yeu_cau_tren_1000_cay
        FROM weekly
        GROUP BY nbhd
        HAVING MAX(so_cay) >= 1000      -- chỉ xét khu phố có từ 1.000 cây để tránh mẫu số quá nhỏ
    )
    SELECT
        'Theo tổng số yêu cầu' AS cach_tinh,
        MAX(tong_so_yeu_cau) AS cao_nhat,
        MIN(tong_so_yeu_cau) AS thap_nhat,
        ROUND(MAX(tong_so_yeu_cau) * 1.0 / MIN(tong_so_yeu_cau), 1) AS so_lan
    FROM nbhd_stats
    UNION ALL
    SELECT
        'Theo số yêu cầu trên 1.000 cây',
        ROUND(MAX(yeu_cau_tren_1000_cay), 1),
        ROUND(MIN(yeu_cau_tren_1000_cay), 1),
        ROUND(MAX(yeu_cau_tren_1000_cay) / MIN(yeu_cau_tren_1000_cay), 1)
    FROM nbhd_stats;
""").df()

gap_sql


Mức chênh lệch giữa khu phố cao nhất và thấp nhất


,cach_tinh,cao_nhat,thap_nhat,so_lan
0,Theo tổng số yêu cầu,13013.0,407.0,32.0
1,Theo số yêu cầu trên 1.000 cây,1688.8,376.2,4.5


**Nhận xét:** Mức chênh lệch giữa khu phố cao nhất và thấp nhất

**Vì sao dùng truy vấn này?** Dùng CTE (`WITH`) tính sẵn số liệu từng khu phố (chỉ xét khu từ 1.000 cây), rồi lấy cao nhất chia thấp nhất theo 2 cách tính, ghép lại bằng `UNION ALL`.

**Kết quả cần đọc:**
- Theo **tổng số yêu cầu**: cao nhất gấp **32 lần** thấp nhất.
- Theo **số yêu cầu trên 1.000 cây**: chỉ còn gấp **4.5 lần**.
- → Phần lớn chênh lệch giữa các khu phố là do **số cây khác nhau**; khi so trên cùng quy mô, khoảng cách thu hẹp đáng kể.

### **5. Mục tiêu:** Nhu cầu bảo trì cây thay đổi như thế nào giữa các ngày trong tuần?

In [8]:
print("\nSố yêu cầu bảo trì cây theo ngày trong tuần (toàn thành phố)")

dow_sql = con.execute("""
    SELECT
        dow,
        CASE dow
            WHEN 0 THEN 'Thứ Hai'
            WHEN 1 THEN 'Thứ Ba'
            WHEN 2 THEN 'Thứ Tư'
            WHEN 3 THEN 'Thứ Năm'
            WHEN 4 THEN 'Thứ Sáu'
            WHEN 5 THEN 'Thứ Bảy'
            WHEN 6 THEN 'Chủ Nhật'
        END AS thu,
        COUNT(*) AS tong_so_yeu_cau,
        ROUND(COUNT(*) * 1.0 / (SELECT COUNT(DISTINCT week_start) FROM weekly), 1) AS tb_moi_ngay
    FROM cases
    WHERE NOT la_trung              -- chỉ đếm yêu cầu không trùng, khớp với biến n
    GROUP BY dow
    ORDER BY dow;
""").df()

dow_sql


Số yêu cầu bảo trì cây theo ngày trong tuần (toàn thành phố)


,dow,thu,tong_so_yeu_cau,tb_moi_ngay
0,0,Thứ Hai,19299,31.4
1,1,Thứ Ba,19797,32.2
2,2,Thứ Tư,19090,31.1
3,3,Thứ Năm,18994,30.9
4,4,Thứ Sáu,17025,27.7
5,5,Thứ Bảy,10714,17.4
6,6,Chủ Nhật,11138,18.1


**Nhận xét:** Nhu cầu theo ngày trong tuần

**Vì sao dùng truy vấn này?** Bảng `weekly` không còn thông tin ngày nên dùng bảng `cases`, `GROUP BY dow` để đếm số yêu cầu từng thứ (chỉ tính yêu cầu không trùng, `WHERE NOT la_trung`); trung bình mỗi ngày = tổng chia 614 tuần.

**Kết quả cần đọc:**
- **Thứ Hai – Thứ Năm** cao nhất (khoảng **31–32 yêu cầu/ngày**), cao nhất là **Thứ Ba** (32.2).
- **Thứ Bảy, Chủ Nhật** chỉ khoảng **17–18 yêu cầu/ngày**, thấp hơn ngày thường khoảng **42%**.
- → Nhiều khả năng do người dân và nhân viên **ít gửi yêu cầu vào cuối tuần**, chứ không phải cây ít hỏng hơn.

### **6. Mục tiêu:** Nhu cầu bảo trì cây thay đổi như thế nào theo tháng trong năm?

In [9]:
print("\nNhu cầu bảo trì cây theo tháng (chỉ tính các năm đủ 12 tháng: 2015-2025)")

month_sql = con.execute("""
    SELECT
        month,
        CAST(SUM(n) AS INTEGER) AS tong_so_yeu_cau,
        ROUND(SUM(n) * 1.0 / COUNT(DISTINCT week_start), 1) AS tb_moi_tuan_toan_tp
    FROM weekly
    WHERE year <= 2025
    GROUP BY month
    ORDER BY month;
""").df()

month_sql


Nhu cầu bảo trì cây theo tháng (chỉ tính các năm đủ 12 tháng: 2015-2025)


,month,tong_so_yeu_cau,tb_moi_tuan_toan_tp
0,1,9351,190.8
1,2,8729,194.0
2,3,8214,171.1
3,4,7636,162.5
4,5,8848,180.6
5,6,9156,194.8
6,7,9465,193.2
7,8,9376,191.3
8,9,8761,186.4
9,10,10030,204.7


**Nhận xét:** Nhu cầu theo tháng

**Vì sao dùng truy vấn này?** `GROUP BY month` tính tổng và trung bình mỗi tuần (toàn thành phố) của từng tháng. Chỉ lấy các năm đủ 12 tháng (**2015–2025**) để so sánh công bằng.

**Kết quả cần đọc:**
- **Tháng 10** cao nhất (**205 yêu cầu/tuần**), tiếp theo là nhóm **tháng 6–8** (khoảng 191–195).
- **Tháng 11–12** thấp nhất, tháng 12 chỉ **146 yêu cầu/tuần** → tháng cao nhất hơn tháng thấp nhất khoảng **40%**.
- → Nhu cầu có tính **mùa vụ rõ rệt**; điểm trũng cuối năm có thể liên quan đến **kỳ nghỉ lễ** (ít người gửi yêu cầu).

### **7. Mục tiêu:** Nhu cầu bảo trì cây thay đổi như thế nào qua các năm?

In [10]:
print("\nNhu cầu bảo trì cây theo từng năm")

year_sql = con.execute("""
    SELECT
        year,
        COUNT(DISTINCT week_start) AS so_tuan,
        CAST(SUM(n) AS INTEGER) AS tong_so_yeu_cau,
        ROUND(SUM(n) * 1.0 / COUNT(DISTINCT week_start), 1) AS tb_moi_tuan_toan_tp
    FROM weekly
    GROUP BY year
    ORDER BY year;
""").df()

year_sql


Nhu cầu bảo trì cây theo từng năm


,year,so_tuan,tong_so_yeu_cau,tb_moi_tuan_toan_tp
0,2015,52,6944,133.5
1,2016,52,7649,147.1
2,2017,52,10135,194.9
3,2018,53,8638,163.0
4,2019,52,9342,179.7
5,2020,52,7635,146.8
6,2021,52,10746,206.7
7,2022,52,7499,144.2
8,2023,52,10694,205.7
9,2024,53,10609,200.2


**Nhận xét:** Xu hướng nhu cầu qua các năm

**Vì sao dùng truy vấn này?** `GROUP BY year` tính tổng và trung bình mỗi tuần của từng năm. Năm 2026 chỉ có 40 tuần nên so sánh bằng **trung bình mỗi tuần**, không so bằng tổng.

**Kết quả cần đọc:**
- Nhu cầu **tăng mạnh** qua các năm: từ **134 yêu cầu/tuần (2015)** lên **300 yêu cầu/tuần (2026)**, gấp khoảng **2.2 lần**.
- **2017 tăng vọt 33%** so với 2016 — trùng thời điểm áp dụng **Prop-E** (thành phố nhận trách nhiệm chăm sóc cây đường phố, đã thấy trong ghi chú 311 ở Notebook 2).
- **2020** (COVID-19) và **2022** giảm, sau đó tăng lại; **2025–2026** tăng mạnh nhất.
- Mức tăng mạnh nhất (2025–2026) đến **khoảng một năm sau** khi hệ thống 311 đổi bộ phân loại yêu cầu (6/2024) và tập trung chủ yếu ở loại **"Cảnh quan"**; bản thân lần đổi hệ thống chủ yếu làm **cơ cấu loại yêu cầu** thay đổi chứ không làm tổng số yêu cầu tăng vọt (Notebook 6).
- → Xu hướng thay đổi theo thời gian là lý do cần đánh giá mô hình trên **nhiều mốc thời gian** và thử **tái huấn luyện định kỳ** (RQ3).

### **8. Mục tiêu:** Những tuần nào có tổng nhu cầu bảo trì cây cao nhất toàn thành phố?

In [11]:
print("\n10 tuần có tổng số yêu cầu cao nhất (toàn thành phố)")

top_weeks_sql = con.execute("""
    SELECT
        week_start,
        CAST(SUM(n) AS INTEGER) AS tong_so_yeu_cau,
        MAX(PRCP) AS luong_mua_tuan_mm,
        MAX(AWND) AS gio_manh_nhat_m_s
    FROM weekly
    GROUP BY week_start
    ORDER BY tong_so_yeu_cau DESC
    LIMIT 10;
""").df()

top_weeks_sql


10 tuần có tổng số yêu cầu cao nhất (toàn thành phố)


,week_start,tong_so_yeu_cau,luong_mua_tuan_mm,gio_manh_nhat_m_s
0,2023-03-20,882,36.3,8.8
1,2021-10-25,873,8.6,5.5
2,2021-10-18,644,139.0,10.2
3,2024-02-05,597,10.4,6.8
4,2024-01-29,516,86.9,12.0
5,2025-12-22,472,75.5,9.7
6,2023-01-02,466,90.4,8.2
7,2017-01-09,446,61.7,9.8
8,2021-01-18,422,5.6,10.6
9,2019-01-14,417,51.2,9.9


**Nhận xét:** Các tuần cao điểm nhất toàn thành phố

**Vì sao dùng truy vấn này?** Gộp 41 khu phố theo `GROUP BY week_start` để tìm các tuần cả thành phố cùng tăng vọt, kèm lượng mưa và gió của tuần đó để đối chiếu.

**Kết quả cần đọc:**
- Tuần cao nhất (**882 yêu cầu**, tuần 20/03/2023) gấp khoảng **4–5 lần** một tuần bình thường (trung bình khoảng 190 yêu cầu/tuần).
- Hầu hết các tuần cao điểm đều có **mưa lớn hoặc gió mạnh** (gió 8–12 m/s, nhiều tuần mưa trên 50 mm).
- Nhu cầu thường tăng mạnh nhất vào **tuần ngay sau bão**: bão tháng 10/2021 mưa 139 mm (tuần 18/10) nhưng tuần kế tiếp mới cao nhất (**873 yêu cầu**); tương tự bão đầu tháng 2/2024.
-  → Các trận **bão cực đoan** có thể kéo nhu cầu tăng mạnh sang cả tuần sau, nhưng **không phải tuần mưa nào cũng vậy** (kiểm định ở Notebook 4 cho thấy mưa nói chung không làm tăng nhu cầu tuần sau).

### **9. Mục tiêu:** Khu phố nào thường xuyên có những tuần không phát sinh yêu cầu nào?

In [12]:
print("\nTỷ lệ tuần không có yêu cầu nào (n = 0) theo khu phố")

zero_weeks_sql = con.execute("""
    SELECT
        nbhd,
        COUNT(*) AS tong_so_tuan,
        CAST(SUM(CASE WHEN n = 0 THEN 1 ELSE 0 END) AS INTEGER) AS so_tuan_khong_co_yeu_cau,
        ROUND(100.0 * SUM(CASE WHEN n = 0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS ty_le_phan_tram
    FROM weekly
    GROUP BY nbhd
    ORDER BY ty_le_phan_tram DESC
    LIMIT 10;
""").df()

zero_weeks_sql


Tỷ lệ tuần không có yêu cầu nào (n = 0) theo khu phố


,nbhd,tong_so_tuan,so_tuan_khong_co_yeu_cau,ty_le_phan_tram
0,Treasure Island,614,564,91.9
1,Lincoln Park,614,539,87.8
2,Presidio,614,530,86.3
3,McLaren Park,614,414,67.4
4,Seacliff,614,337,54.9
5,Lakeshore,614,277,45.1
6,Japantown,614,274,44.6
7,Golden Gate Park,614,254,41.4
8,Chinatown,614,246,40.1
9,Visitacion Valley,614,205,33.4


**Nhận xét:** Tỷ lệ tuần không có yêu cầu theo khu phố

**Vì sao dùng truy vấn này?** Dùng `SUM(CASE WHEN n = 0 ...)` để đếm số tuần khu phố không có yêu cầu nào, rồi chia cho 614 tuần để ra tỷ lệ phần trăm.

**Kết quả cần đọc:**
- **Treasure Island, Lincoln Park, Presidio** gần như luôn "yên tĩnh": **86–92%** số tuần không có yêu cầu nào.
- Toàn bộ dữ liệu chỉ có **19.4%** tuần bằng 0, nhưng các tuần này **tập trung ở một nhóm nhỏ** khu phố — chủ yếu là khu công viên hoặc khu có **ít cây đường phố** (kể cả khu đông dân như Chinatown, Japantown).
- → Ở những khu phố này nhu cầu rất **thưa và khó đoán**; vì vậy Notebook 7 báo cáo sai số riêng cho từng nhóm khu phố (thấp, trung bình, cao) và dùng thêm thước đo MASE.

### **10. Mục tiêu:** Tạo đặc trưng trễ và trung bình trượt phục vụ dự báo (Notebook 5, 7 và 8)

Quy ước: dự báo được đưa ra **vào cuối tuần hiện tại**, cho **tuần tới**. Vì vậy mọi đặc trưng chỉ được dùng số liệu của tuần hiện tại và các tuần trước đó.

| Cột | Ý nghĩa |
|---|---|
| `n` | Số yêu cầu của tuần hiện tại (cũng là dự báo của mốc "lặp lại tuần trước") |
| `n_lag1`, `n_lag2` | Số yêu cầu của 1 tuần và 2 tuần trước |
| `n_lag52` | Số yêu cầu cùng kỳ năm trước của tuần hiện tại (52 tuần trước) |
| `n_lag51` | Số yêu cầu cùng kỳ năm trước của **tuần tới** (51 tuần trước tuần hiện tại) — dự báo của mốc "cùng tuần năm trước" |
| `n_ma4` | Trung bình 4 tuần gần nhất, tính cả tuần hiện tại (cũng là dự báo của mốc "trung bình 4 tuần") |
| `y_next` | **Biến mục tiêu chính**: số yêu cầu của tuần tới |

Hai biến đếm phụ được tạo đặc trưng theo cùng quy tắc, thêm tiền tố tên biến:
- `n_all_lag1`, `n_all_lag2`, `n_all_lag52`, `n_all_lag51`, `n_all_ma4` và biến mục tiêu `y_next_all` (giữ cả yêu cầu trùng).
- `n_damaged_lag1`, `n_damaged_lag2`, `n_damaged_lag52`, `n_damaged_lag51`, `n_damaged_ma4` và biến mục tiêu `y_next_damaged` (chỉ loại "cây hư hại").

In [13]:
features_sql = con.execute("""
    SELECT
        *,
        -- Biến mục tiêu chính n (đã loại yêu cầu trùng)
        LAG(n, 1)  OVER w AS n_lag1,
        LAG(n, 2)  OVER w AS n_lag2,
        LAG(n, 52) OVER w AS n_lag52,
        LAG(n, 51) OVER w AS n_lag51,
        AVG(n)     OVER (w ROWS BETWEEN 3 PRECEDING AND CURRENT ROW) AS n_ma4,
        LEAD(n, 1) OVER w AS y_next,

        -- Biến phụ n_all (giữ cả yêu cầu trùng)
        LAG(n_all, 1)  OVER w AS n_all_lag1,
        LAG(n_all, 2)  OVER w AS n_all_lag2,
        LAG(n_all, 52) OVER w AS n_all_lag52,
        LAG(n_all, 51) OVER w AS n_all_lag51,
        AVG(n_all)     OVER (w ROWS BETWEEN 3 PRECEDING AND CURRENT ROW) AS n_all_ma4,
        LEAD(n_all, 1) OVER w AS y_next_all,

        -- Biến phụ n_damaged (chỉ loại "cây hư hại")
        LAG(n_damaged, 1)  OVER w AS n_damaged_lag1,
        LAG(n_damaged, 2)  OVER w AS n_damaged_lag2,
        LAG(n_damaged, 52) OVER w AS n_damaged_lag52,
        LAG(n_damaged, 51) OVER w AS n_damaged_lag51,
        AVG(n_damaged)     OVER (w ROWS BETWEEN 3 PRECEDING AND CURRENT ROW) AS n_damaged_ma4,
        LEAD(n_damaged, 1) OVER w AS y_next_damaged
    FROM weekly
    WINDOW w AS (PARTITION BY nbhd ORDER BY week_start)
    ORDER BY nbhd, week_start;
""").df()

print(features_sql.shape)
features_sql[features_sql["nbhd"] == "Mission"][["nbhd", "week_start", "n", "n_lag1", "n_lag2", "n_ma4", "y_next"]].head(6)

(25174, 33)


,nbhd,week_start,n,n_lag1,n_lag2,n_ma4,y_next
11052,Mission,2015-01-05,13,<NA>,<NA>,13.000000,18
11053,Mission,2015-01-12,18,13,<NA>,15.500000,21
11054,Mission,2015-01-19,21,18,13,17.333333,26
11055,Mission,2015-01-26,26,21,18,19.500000,53
11056,Mission,2015-02-02,53,26,21,29.500000,44
11057,Mission,2015-02-09,44,53,26,36.000000,38


In [14]:
# Kiểm tra n_lag51: giá trị ở tuần t phải bằng y_next của tuần cách đó 52 tuần về trước
# (tức là số yêu cầu của cùng tuần năm trước so với tuần tới)
m = features_sql[features_sql["nbhd"] == "Mission"].reset_index(drop=True)
kiem_tra = (m["n_lag51"].iloc[52:].values == m["y_next"].shift(52).iloc[52:].values).all()
print("n_lag51 khớp với y_next của 52 tuần trước:", kiem_tra)

# Kiểm tra biến mục tiêu: tổng y_next phải bằng tổng biến đếm trừ tuần đầu tiên của mỗi khu phố
# (tuần đầu tiên không phải là "tuần tới" của tuần nào)
tuan_dau = features_sql["week_start"] == features_sql["week_start"].min()
for dem, muc_tieu in [("n", "y_next"), ("n_all", "y_next_all"), ("n_damaged", "y_next_damaged")]:
    print(f"{muc_tieu:15s}: {int(features_sql[muc_tieu].sum()):,} = {int(features_sql.loc[~tuan_dau, dem].sum()):,}")

n_lag51 khớp với y_next của 52 tuần trước: True
y_next         : 115,917 = 115,917
y_next_all     : 132,055 = 132,055
y_next_damaged : 44,669 = 44,669


**Nhận xét:** Tạo đặc trưng trễ và trung bình trượt

**Vì sao dùng truy vấn này?** Dùng hàm cửa sổ `PARTITION BY nbhd ORDER BY week_start` để tính riêng cho từng khu phố theo thứ tự thời gian: `LAG` lấy số yêu cầu các tuần trước, `AVG ... ROWS BETWEEN 3 PRECEDING AND CURRENT ROW` tính trung bình 4 tuần gần nhất, `LEAD` lấy số yêu cầu tuần sau làm biến mục tiêu. Cùng một quy tắc được áp dụng cho ba biến đếm `n`, `n_all` và `n_damaged`.

**Kết quả cần đọc:**
- Bảng đặc trưng có **25,174 dòng, 33 cột** (15 cột gốc và 18 cột mới: 6 cột cho mỗi biến đếm).
- Kiểm tra trên Mission: `n_lag1` của mỗi tuần đúng bằng `n` của tuần trước, `y_next` đúng bằng `n` của tuần sau, `n_ma4` đúng bằng trung bình 4 tuần gần nhất, và `n_lag51` đúng bằng số yêu cầu cùng tuần năm trước của tuần tới.
- Các tuần đầu thiếu đặc trưng trễ và tuần cuối thiếu biến mục tiêu — Notebook 5 và 7 sẽ bỏ các dòng này khi huấn luyện.

## Lưu kết quả

In [15]:
os.makedirs("../Outputs/tables", exist_ok=True)

sql_results = {
    "sql_01_top_nbhd.csv": top_nbhd_sql,
    "sql_02_bottom_nbhd.csv": bottom_nbhd_sql,
    "sql_03_per_1000_trees.csv": per_tree_sql,
    "sql_04_gap_high_low.csv": gap_sql,
    "sql_05_day_of_week.csv": dow_sql,
    "sql_06_month.csv": month_sql,
    "sql_07_year.csv": year_sql,
    "sql_08_top_weeks.csv": top_weeks_sql,
    "sql_09_zero_weeks.csv": zero_weeks_sql,
}

for filename, result_df in sql_results.items():
    result_df.to_csv(f"../Outputs/tables/{filename}", index=False)

# Bảng đặc trưng dùng cho Notebook 5, 7 và 8
features_sql.to_csv("../Data/processed/SF_Tree_Weekly_Features.csv", index=False)

print("Đã lưu xong 9 bảng kết quả SQL và bảng đặc trưng cho Notebook 5, 7 và 8.")

Đã lưu xong 9 bảng kết quả SQL và bảng đặc trưng cho Notebook 5, 7 và 8.


In [16]:
con.close()
print("\nĐã đóng kết nối DuckDB.")


Đã đóng kết nối DuckDB.
